# 01_workload_distributor_with_weather
Enhanced pipeline with real weather from Open-Meteo, improved thermal/water model, dynamic water model.

In [1]:
import pandas as pd
import numpy as np
import os
import requests
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

base_dir = r"C:\Users\RANADEEP\Documents\IBM Project\Alibaba_v2018"
raw_dir = os.path.join(base_dir, "01_raw_data")
processed_dir = os.path.join(base_dir, "02_processed_data")
weather_dir = os.path.join(base_dir, "03_weather_data")

os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)
os.makedirs(weather_dir, exist_ok=True)


In [2]:
# Weather Fetcher with error handling
DC_LATITUDE = 12.9716
DC_LONGITUDE = 77.5946
start_date = "2018-01-01"
end_date = "2018-01-08"

print('Fetching weather from Open-Meteo...')
url = f"https://archive-api.open-meteo.com/v1/archive?latitude={DC_LATITUDE}&longitude={DC_LONGITUDE}&start_date={start_date}&end_date={end_date}&hourly=temperature_2m,relative_humidity_2m"

try:
    res = requests.get(url, timeout=30)
    res.raise_for_status()
    weather_data = res.json()
    df_weather = pd.DataFrame({
        'time': pd.to_datetime(weather_data['hourly']['time']),
        'temperature_2m': weather_data['hourly']['temperature_2m'],
        'humidity_2m': weather_data['hourly']['relative_humidity_2m']
    })
except Exception as e:
    print(f"Weather API failed: {e}. Using synthetic weather.")
    df_weather = pd.DataFrame({
        'time': pd.date_range(start=start_date, end=end_date, freq='H'),
        'temperature_2m': 25.0,
        'humidity_2m': 50.0
    })

df_weather.to_csv(os.path.join(weather_dir, 'historical_weather.csv'), index=False)
print('Weather data saved.')


Fetching weather from Open-Meteo...
Weather data saved.


In [3]:
# Load baseline workload
ts_df = pd.read_csv(os.path.join(processed_dir, 'workload_timeseries.csv'))

# Merge with weather
df_weather['datetime'] = df_weather['time']
ts_df['datetime'] = pd.to_datetime(ts_df['datetime'])
df_weather['datetime'] = pd.to_datetime(df_weather['datetime'])
merged_df = pd.merge_asof(ts_df.sort_values('datetime'), df_weather.sort_values('datetime'), on='datetime', direction='nearest')

# Enhanced dynamic model
cooling_power_kw = (merged_df['temperature_2m'] - 20).clip(lower=0) * 0.05
merged_df['total_power_kw'] = merged_df['power_kw'] + cooling_power_kw

dynamic_wue = 1.5 + (merged_df['temperature_2m'] - 20).clip(lower=0) * 0.02 - (merged_df['humidity_2m'] - 50) * 0.005
merged_df['dynamic_water_liters'] = merged_df['total_power_kw'] * dynamic_wue

merged_df.to_csv(os.path.join(processed_dir, 'workload_timeseries_enhanced.csv'), index=False)
print('Enhanced dataset saved.')


Enhanced dataset saved.


In [4]:
# Train enhanced predictors with CORRECT train_test_split usage
features = merged_df[['cpu_util_percent', 'temperature_2m', 'humidity_2m']]
y_power = merged_df['total_power_kw']
y_water = merged_df['dynamic_water_liters']

indices = np.arange(len(features))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
y_power_train, y_power_test = y_power.iloc[train_idx], y_power.iloc[test_idx]
y_water_train, y_water_test = y_water.iloc[train_idx], y_water.iloc[test_idx]

model_power = RandomForestRegressor(n_estimators=50, random_state=42)
model_power.fit(X_train, y_power_train)
pred_power = model_power.predict(X_test)

model_water = RandomForestRegressor(n_estimators=50, random_state=42)
model_water.fit(X_train, y_water_train)
pred_water = model_water.predict(X_test)

print(f"Enhanced Power MAE: {mean_absolute_error(y_power_test, pred_power):.4f} kW")
print(f"Enhanced Water MAE: {mean_absolute_error(y_water_test, pred_water):.4f} Liters")


Enhanced Power MAE: 0.0001 kW
Enhanced Water MAE: 0.0003 Liters
